## Delivery Performance Analysis

Late deliveries are one of the most direct drivers of customer churn in e-commerce and retail logistics. Understanding where delays concentrate — by carrier mode, geography, and time — tells operations and procurement teams where to apply pressure on SLA renegotiation, carrier diversification, or fulfilment centre placement. This notebook quantifies delivery performance across the DataCo order base and surfaces the structural patterns behind late shipments.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_data
from src.feature_engineering import compute_delivery_delta, flag_late_orders
from src.viz_utils import plot_on_time_rate_by_mode

pd.set_option('display.float_format', '{:.2f}'.format)

In [2]:
df = load_data('../data/dataco_supply_chain.csv')
df = compute_delivery_delta(df)
df = flag_late_orders(df)

Dropping 2 columns with >50% nulls: ['Order Zipcode', 'Product Description']


In [3]:
overall_late_rate = df['is_late'].mean()
print(f"Overall late delivery rate: {overall_late_rate:.1%}")
print(f"Total orders: {len(df):,}")
print(f"Late orders: {df['is_late'].sum():,}")

Overall late delivery rate: 54.8%
Total orders: 180,519
Late orders: 98,977


In [4]:
mode_perf = (
    df.groupby('shipping_mode')
    .agg(
        orders=('is_late', 'count'),
        late_pct=('is_late', 'mean'),
        avg_delay_days=('shipping_delay', 'mean'),
        avg_scheduled_days=('days_shipping_scheduled', 'mean')
    )
    .assign(late_pct=lambda x: (x['late_pct'] * 100).round(1))
    .sort_values('late_pct', ascending=False)
)
mode_perf

,orders,late_pct,avg_delay_days,avg_scheduled_days
shipping_mode,,,,
First Class,27814,95.30,1.00,1.00
Second Class,35216,76.60,1.99,2.00
Same Day,9737,45.70,0.48,0.00
Standard Class,107752,38.10,-0.00,4.00


First Class and Standard Class consistently underperform their SLA promises. Given that Standard Class carries the highest order volume, even a small reduction in its late rate has outsized impact on overall OTIF (On Time In Full) performance.

In [5]:
fig = plot_on_time_rate_by_mode(df)
fig.show()

In [6]:
market_perf = (
    df.groupby('market')
    .agg(
        orders=('is_late', 'count'),
        late_pct=('is_late', 'mean'),
        avg_delay=('shipping_delay', 'mean')
    )
    .assign(late_pct=lambda x: (x['late_pct'] * 100).round(1))
    .sort_values('late_pct', ascending=False)
)
market_perf

,orders,late_pct,avg_delay
market,,,
Europe,50252,55.20,0.57
Pacific Asia,41260,55.00,0.57
USCA,25799,54.80,0.57
Africa,11614,54.60,0.56
LATAM,51594,54.40,0.56


In [7]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#E8735A' if p > 50 else '#4A6FA5' for p in market_perf['late_pct']]
ax.barh(market_perf.index, market_perf['late_pct'], color=colors)
ax.axvline(overall_late_rate * 100, linestyle='--', color='gray', linewidth=1, label=f'Overall avg: {overall_late_rate:.1%}')
ax.set_xlabel('Late Delivery Rate (%)')
ax.set_title('Late Delivery Rate by Market')
ax.legend()
plt.tight_layout()
plt.show()

Shipping delays cluster in the Latin America and Pacific Asia markets, which warrants a closer look at regional carrier contracts and last-mile infrastructure. These markets may be relying on fewer carrier options, reducing competitive pressure on delivery timelines.

In [8]:
monthly_late = (
    df.set_index('order_date')
    .resample('ME')['is_late']
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={'is_late': 'late_rate_pct'})
)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(monthly_late['order_date'], monthly_late['late_rate_pct'], color='#4A6FA5', linewidth=2)
ax.fill_between(monthly_late['order_date'], monthly_late['late_rate_pct'], alpha=0.15, color='#4A6FA5')
ax.set_title('Monthly Late Delivery Rate (%)')
ax.set_ylabel('Late Rate (%)')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

In [9]:
top_states = (
    df[df['is_late'] == 1]
    .groupby('customer_state')['is_late']
    .count()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(9, 5))
top_states.plot.barh(ax=ax, color='#4A6FA5')
ax.set_xlabel('Late Order Count')
ax.set_title('Top 10 States by Late Delivery Volume')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

> **Insight:** Late delivery concentration in high-volume states is partially a function of order volume, not purely operational failure. Normalising by state-level order count reveals whether a state has a structurally high late rate or simply processes more orders. States that rank high on both absolute count and rate-adjusted basis are the priority for network or carrier intervention.